# Voxelising a Laguerre diagram

In this notebook we test the speed and accuracy of the method `synthetmic.LaguerreDiagramGenerator.voxelise` from 
[SynthetMic](https://github.com/synthetic-microstructures/synthetmic). For every voxel in a cubic mesh, we compute which Laguerre cell (grain) it belongs to. We give examples of periodic and non-periodic Laguerre diagrams with 10,000 cells, and we show that for a cubic mesh with 1,000,000 voxels, the method takes less than 1 second. This method uses k-d trees for fast computation. To test the accuracy of this function, we use a simple brute-force method as a benchmark. 

Note that the SynthetMic library can be installed via pip as follows: `pip install synthetmic`.

First we import the relevant libraries:

In [ ]:
import numpy as np
from synthetmic import LaguerreDiagramGenerator, DiagramConfig
from synthetmic.plot import plot_cells_as_pyvista_fig
from synthetmic.utils import compute_cell_centers
from pysdot import PowerDiagram
import time

The following is a very slow, brute-force method for voxelising a fully periodic diagram (periodic in every dimension), which we will use for testing the method `synthetmic.LaguerreDiagramGenerator.voxelise`:

In [ ]:
def _tile_positions(positions: np.ndarray, boxsize: np.ndarray) -> np.ndarray:
    positions = np.asarray(positions)
    boxsize = np.asarray(boxsize)
    dimensions = positions.shape[1]

    tiled = positions.copy()

    for i in range(dimensions):
        # Create a displacement vector for the current dimension
        # e.g., for i=0 in 3D: [L1, 0, 0]
        offset = np.zeros(dimensions)
        offset[i] = boxsize[i]

        # Stack the current tiled block with a version shifted
        # negatively and a version shifted positively
        tiled = np.vstack((tiled - offset, tiled, tiled + offset))

    return tiled


def _tile_weights(weights: np.ndarray, dimensions: int) -> np.ndarray:
    return np.tile(weights, reps=3**dimensions)


def _bruteforce_voxelise_periodic_diagram(
    points: np.ndarray, pd: PowerDiagram, box: np.ndarray
):

    x = pd.get_positions()
    N, D = x.shape
    periodic_seeds = _tile_positions(positions=x, boxsize=box[:, 1] - box[:, 0])

    w = pd.get_weights()
    periodic_weights = _tile_weights(weights=w, dimensions=D)

    num_points = np.size(points, 0)
    grain_indices = [0] * num_points

    for i in range(num_points):
        p = points[i]
        grain_indices[i] = (
            np.argmin(
                np.sum((periodic_seeds - p) * (periodic_seeds - p), axis=1)
                - periodic_weights
            )
            % N
        )

    return np.asarray(grain_indices, dtype=np.int32)

## 1. Test `synthetmic.LaguerreDiagramGenerator.voxelise` for periodic diagrams

In this section we test `synthetmic.LaguerreDiagramGenerator.voxelise` for voxelising periodic Laguerre diagrams. First we create a small example for accuracy testing (a 3D periodic Laguerre diagram with 1000 grains, where the volumes of the grains are drawn from a lognormal distribution):

In [ ]:
# Create a periodic Laguerre diagram with 1000 grains

corner = np.array([-1,1,0]) # coordinates of the bottom corner of the box
Lx = 2. # length of the box in the x-direction 
Ly = 3. # length of the box in the y-direction
Lz = 1. # length of the box in the z-direction
box = np.array([[corner[0], corner[0]+Lx],
                [corner[1], corner[1]+Ly],
                [corner[2], corner[2]+Lz]])
periodicity = [True,True,True]

# Number of grains
N = 1_000

ln_mean = 1 # mean of the lognormal distribution
std_dev = 0.5 # standard deviation of the lognormal distribution

# Lognormal parameters (see here https://en.wikipedia.org/wiki/Log-normal_distribution):
Sigma = np.sqrt(np.log(1+(std_dev/ln_mean)**2))
Mu = -0.5*Sigma**2+np.log(ln_mean)

radii = np.random.lognormal(Mu,Sigma,N) # Draw N radii from a lognormal distribution
target_vols = 4/3*np.pi*radii**3 # Compute the corresponding sphere volumes
target_vols  = target_vols*Lx*Ly*Lz/np.sum(target_vols) # Normalise the volumes so that the total volume equals the volume of the box

# Remark: Only the ratio std_dev/ln_mean matters here. E.g., taking (ln_mean,std_dev)=(1,0.35) gives exactly the same result (up to
# machine precision) as taking (ln_mean,std_dev)=(c,0.35*c) for any constant c. This is because the volumes of the grains are normalised.

percent_tol = 1. # Volume tolerance: maximum percentage error of the volumes of the grains
num_Lloyd = 5 # Number of Lloyd iterations. The Lloyd iterations regularise the microstructure
seeds = corner + np.random.rand(N,3)@np.diag([Lx,Ly,Lz]) # Initial seed locations (drawn at random from the uniform distribution on the box)

# Initialise a LaguerreDiagramGenerator object
diagram = LaguerreDiagramGenerator(tol=percent_tol, n_iter=num_Lloyd) 

# Call the fit method to generate a Laguerre diagram with grains of volumes target_vols (to within the specified tolerance percent_tol)
config = DiagramConfig(
        seeds=seeds,
        volumes=target_vols,
        domain=box,
        periodic=periodicity,
)
start = time.time()
diagram.fit(config)
end = time.time()
print(f'Run time = {end-start} seconds') 

Plot the diagram:

In [ ]:
pl = plot_cells_as_pyvista_fig(generator=diagram, colorby=diagram.get_fitted_volumes(), notebook=True)
pl.show()

Next we create a small mesh ($50^3$ grid points) for accuracy testing:

In [ ]:
Nx = 50 # number of grid points in the x-direction
Ny = 50 # number of grid points in the y-direction
Nz = 50 # number of gird points in the z-direction
points_per_dim = (Nx, Ny, Nz)
points = compute_cell_centers(
    origin=config.domain[:, 0],
    size=config.domain[:, 1] - config.domain[:, 0],
    points_per_dim=points_per_dim,
)
points = points.reshape(-1, 3)

Now we voxelise the diagram. Note that the `voxelise` method returns a `synthetmic.data.utils.VoxelGrid` object. This contains three fields, namely:

- `voxels`, a numpy array of shape `points_per_dim` that contains the grain ids that each voxel belongs to;
- `origin`, a sequence of floats of length `len(points_per_dim)` depicting the origin of the diagram domain;
- `size`, a sequence of floats of length `len(points_per_dim)` depicting the length of the domain in each direction.

In [ ]:
start = time.time()
voxel_grid = diagram.voxelise(points_per_dim=points_per_dim, domain=config.domain)
end = time.time()
print(f'Run time = {end-start} seconds')
voxel_grid

If we want the grain ids as a flattened array of shape `(Nx*Ny*Nz,)`, we just get the `voxels` field and flatten it accordingly.

In [ ]:
grain_indices_KDTree = voxel_grid.voxels.flatten(order="F") 
# grain_indices_KDTree[k] is the index of the grain containing mesh point k
print(grain_indices_KDTree) 

As an accuracy test, we voxelise the diagram using the brute-force approach, which is much slower:

In [ ]:
start = time.time()
grain_indices_bruteforce = _bruteforce_voxelise_periodic_diagram(
        points=points, pd=diagram.pd_, box=config.domain
    )
end = time.time()
print(f'Run time = {end-start} seconds') 
# grain_indices_bruteforce[k] is the index of the grain containing mesh point k
print(grain_indices_bruteforce) 

Check that the two methods give the same answer:

In [ ]:
# This prints 0 if both methods give the same answer.
print(np.size(np.argwhere(grain_indices_KDTree != grain_indices_bruteforce)))

Both methods give the same answer, but the `voxelise` method is much quicker!

Now we do a large test with 10,000 grains to test the speed of `voxelise`. First we generate the diagram using Algorithm 2 from [this paper](https://www.tandfonline.com/doi/full/10.1080/14786435.2020.1790053) (this takes a couple of minutes): 

In [ ]:
# Create a 3D periodic Laguerre diagram with 10,000 grains

corner = np.array([0,0,0]) # coordinates of the bottom corner of the box
Lx = 2. # length of the box in the x-direction 
Ly = 2. # length of the box in the y-direction
Lz = 2. # length of the box in the z-direction
box = np.array([[corner[0], corner[0]+Lx],
                [corner[1], corner[1]+Ly],
                [corner[2], corner[2]+Lz]])
periodicity = [True,True,True]

# Number of grains
N = 10_000

ln_mean = 1 # mean of the lognormal distribution
std_dev = 0.35 # standard deviation of the lognormal distribution

# Lognormal parameters (see here https://en.wikipedia.org/wiki/Log-normal_distribution):
Sigma = np.sqrt(np.log(1+(std_dev/ln_mean)**2))
Mu = -0.5*Sigma**2+np.log(ln_mean)

radii = np.random.lognormal(Mu,Sigma,N) # Draw N radii from a lognormal distribution
target_vols = 4/3*np.pi*radii**3 # Compute the corresponding sphere volumes
target_vols  = target_vols*Lx*Ly*Lz/np.sum(target_vols) # Normalise the volumes so that the total volume equals the volume of the box

# Remark: Only the ratio std_dev/ln_mean matters here. E.g., taking (ln_mean,std_dev)=(1,0.35) gives exactly the same result (up to
# machine precision) as taking (ln_mean,std_dev)=(c,0.35*c) for any constant c. This is because the volumes of the grains are normalised.

percent_tol = 1. # Volume tolerance: maximum percentage error of the volumes of the grains
num_Lloyd = 5 # Number of Lloyd iterations. The Lloyd iterations regularise the microstructure
seeds = np.random.rand(N,3)@np.diag([Lx,Ly,Lz]) # Initial seed locations (drawn at random from the uniform distribution on the box)

# Initialise a LaguerreDiagramGenerator object
diagram = LaguerreDiagramGenerator(tol=percent_tol, n_iter=num_Lloyd) 

# Call the fit method to generate a Laguerre diagram with grains of volumes target_vols (to within the specified tolerance percent_tol)
config = DiagramConfig(
        seeds=seeds,
        volumes=target_vols,
        domain=box,
        periodic=periodicity,
)
start = time.time()
diagram.fit(config)
end = time.time()
print(f'Run time = {end-start} seconds') 

Plot the diagram:

In [ ]:
pl = plot_cells_as_pyvista_fig(generator=diagram, colorby=diagram.get_fitted_volumes(), notebook=True)
pl.show()

Create a large mesh (1,000,000 voxels):

In [ ]:
Nx = 100 # number of grid points in the x-direction
Ny = 100 # number of grid points in the y-direction
Nz = 100 # number of gird points in the z-direction
points_per_dim = (Nx, Ny, Nz)

Run time test:

In [ ]:
start = time.time()
voxel_grid = diagram.voxelise(points_per_dim=points_per_dim, domain=config.domain)
end = time.time()
print(f'Run time = {end-start} seconds') 
grain_indices_KDTree = voxel_grid.voxels.flatten(order="F")

Less than 1 second for 10,000 grains and 1,000,000 voxels!

## 2. Test `voxelise` method for non-periodic diagrams

Now we test `voxelise` for non-periodic diagrams.

First we create a large example for testing (a 3D lognormal microstructure with 10,000 grains, as above, but non-periodic):

In [ ]:
# Create a 3D non-periodic Laguerre diagram with 10,000 grains

corner = np.array([0,0,0]) # coordinates of the bottom corner of the box
Lx = 2. # length of the box in the x-direction 
Ly = 2. # length of the box in the y-direction
Lz = 2. # length of the box in the z-direction
box = np.array([[corner[0], corner[0]+Lx],
                [corner[1], corner[1]+Ly],
                [corner[2], corner[2]+Lz]])

# Number of grains
N = 10_000

ln_mean = 1 # mean of the lognormal distribution
std_dev = 0.35 # standard deviation of the lognormal distribution

# Lognormal parameters (see here https://en.wikipedia.org/wiki/Log-normal_distribution):
Sigma = np.sqrt(np.log(1+(std_dev/ln_mean)**2))
Mu = -0.5*Sigma**2+np.log(ln_mean)

radii = np.random.lognormal(Mu,Sigma,N) # Draw N radii from a lognormal distribution
target_vols = 4/3*np.pi*radii**3 # Compute the corresponding sphere volumes
target_vols  = target_vols*Lx*Ly*Lz/np.sum(target_vols) # Normalise the volumes so that the total volume equals the volume of the box

# Remark: Only the ratio std_dev/ln_mean matters here. E.g., taking (ln_mean,std_dev)=(1,0.35) gives exactly the same result (up to
# machine precision) as taking (ln_mean,std_dev)=(c,0.35*c) for any constant c. This is because the volumes of the grains are normalised.

percent_tol = 1. # Volume tolerance: maximum percentage error of the volumes of the grains
num_Lloyd = 5 # Number of Lloyd iterations. The Lloyd iterations regularise the microstructure
seeds = corner + np.random.rand(N,3)@np.diag([Lx,Ly,Lz]) # Initial seed locations (drawn at random from the uniform distribution on the box)

# Initialise a LaguerreDiagramGenerator object
diagram = LaguerreDiagramGenerator(tol=percent_tol, n_iter=num_Lloyd) 

# Call the fit method to generate a Laguerre diagram with grains of volumes target_vols (to within the specified tolerance percent_tol)
config = DiagramConfig(
        seeds=seeds,
        volumes=target_vols,
        domain=box
)
start = time.time()
diagram.fit(config)
end = time.time()
print(f'Run time = {end-start} seconds') 

Plot the diagram:

In [ ]:
pl = plot_cells_as_pyvista_fig(generator=diagram, colorby=diagram.get_fitted_volumes(), notebook=True)
pl.show()

Create a large mesh (1,000,000 voxels):

In [ ]:
Nx = 100 # number of grid points in the x-direction
Ny = 100 # number of grid points in the y-direction
Nz = 100 # number of gird points in the z-direction
points_per_dim = (Nx, Ny, Nz)

Test how long it takes to mesh the diagram:

In [ ]:
start = time.time()
# Note that for non-periodic diagrams we do not pass voxelise the domain
voxel_grid = diagram.voxelise(points_per_dim=points_per_dim)
end = time.time()
print(f'Run time = {end-start} seconds') 
voxel_grid

Less than 1/2 second for 10,000 grains and 1,000,000 voxels!